# Quickstart: распаковка данных и запуск Хоукс-модели

Ноутбук распаковывает дневную сетку активности из `*.csv.gz` и прогоняет главный
эксперимент работы — **Scaled-baseline Hawkes** (полураспады 1 и 3 дня) на целевой
величине `to_ord` (число заказов за день).

Запускать из корня репозитория с активированным окружением (см. README → «Установка»).

## 1. Распаковка данных

В репозитории лежит сжатый `dayuses_cohort_10000_seed42_daily_grid.csv.gz` (~13 МБ).
Распаковываем его рядом в `.csv`, если он ещё не распакован.

In [ ]:
import gzip
import shutil
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

data_dir = ROOT / "data" / "processed" / "orbitals"
csv_gz = data_dir / "dayuses_cohort_10000_seed42_daily_grid.csv.gz"
csv_path = data_dir / "dayuses_cohort_10000_seed42_daily_grid.csv"

if not csv_path.exists():
    with gzip.open(csv_gz, "rb") as src, open(csv_path, "wb") as dst:
        shutil.copyfileobj(src, dst)
    print(f"Распаковано: {csv_path}")
else:
    print(f"Уже распаковано: {csv_path}")

print(f"Размер: {csv_path.stat().st_size / 1e6:.0f} МБ")

## 2. Беглый взгляд на данные

In [ ]:
import pandas as pd

df = pd.read_csv(csv_path, parse_dates=["event_date"])
print(f"строк: {len(df):,}  пользователей: {df['user_id'].nunique():,}")
print(f"окно: {df['event_date'].min().date()} .. {df['event_date'].max().date()}")
df.head()

## 3. Запуск Scaled-baseline Hawkes

Скрипт `scripts/compute/run_experimental_1_hawkes.py` со значениями по умолчанию
воспроизводит числа из работы. Артефакты (графики, таблицы, `summary.json`)
складываются в `diploma/reports/experimental_1_hawkes/`.

In [ ]:
import subprocess
import sys

proc = subprocess.run(
    [sys.executable, "scripts/compute/run_experimental_1_hawkes.py"],
    cwd=ROOT,
    capture_output=True,
    text=True,
)
print(proc.stdout[-2000:])
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr[-2000:])

## 4. Результат

Сравниваем test NLL персонализированного пуассоновского базлайна и Хоукса.
Ожидаемые значения: Personalized GP ≈ 0.4096, Scaled-baseline Hawkes ≈ 0.3958,
подобранные параметры (α, β) ≈ (0.88, 0.88).

In [ ]:
import json

summary = json.loads((ROOT / "diploma/reports/experimental_1_hawkes/summary.json").read_text())

nll_poisson = summary["test_metrics_personalized_poisson"]["mean_poisson_nll"]
nll_hawkes = summary["test_metrics_hawkes"]["mean_poisson_nll"]

print(f"test NLL, Personalized Poisson: {nll_poisson:.4f}")
print(f"test NLL, Scaled-baseline Hawkes: {nll_hawkes:.4f}")
print(f"улучшение: {nll_poisson - nll_hawkes:+.4f}")
print(f"learned base scale: {summary['learned_base_scale']:.4f}")

In [ ]:
from IPython.display import Image

Image(filename=str(ROOT / "diploma/reports/experimental_1_hawkes/user_ll_gain_hist.png"))